## Configuration and Environment Variables

Polars lets you tune how it runs and how its outputs are displayed by setting environment variables before starting your Python session. For example:

- **POLARS_MAX_THREADS**: Limit the number of threads.  
- **POLARS_FMT_MAX_COLS**: Control the number of columns printed.  
- **POLARS_FMT_TABLE_WIDTH**: Adjust DataFrame display width.  

Here’s an example of how you would set these variables in a shell before running Python:

## Generating a Sample CSV Dataset for Demonstration

This script creates a dataset of 100 transactions with random amounts, customer IDs, and transaction dates.

In [4]:
import csv
import random
from datetime import datetime, timedelta

# Define column names
columns = ["transaction_id", "customer_id", "amount", "transaction_date"]

# Generate synthetic rows
rows = []
start_date = datetime(2023, 1, 1)
for i in range(1, 101):
    customer_id = random.randint(1, 10)
    amount = round(random.uniform(10, 2000), 2)
    date = start_date + timedelta(days=random.randint(0, 90))
    rows.append([i, customer_id, amount, date.strftime("%Y-%m-%d")])

# Write to CSV
with open("transactions-polars.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(columns)
    writer.writerows(rows)

## Installation

In [5]:
!uv add polars

Resolved 21 packages in 887ms                                        
Prepared 2 packages in 22.53s                                                polars-runtime-32    ------------------------------ 48.94 MiB/48.98 MiB         polars-runtime-32    ------------------------------ 512.00 KiB/48.98 MiB        
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 351ms2.1                            
 + polars==1.42.1
 + polars-runtime-32==1.42.1


In [6]:
import polars as pl

print(pl.__version__)

1.42.1


## Series: 1D arrays with a defined type

A Series in Polars is a one-dimensional array, similar to a column in a spreadsheet or database table. Each Series has:

- A name (the column label).  
- A data type (such as Int64, Utf8, Float64, Boolean, etc.).  
- An ordered collection of values.

In [7]:
import polars as pl
# Create a Series of integers
s = pl.Series("numbers", [1, 2, 3, 4, 5])
print(s)

shape: (5,)
Series: 'numbers' [i64]
[
	1
	2
	3
	4
	5
]


## DataFrame: A collection of series

A DataFrame is a two-dimensional structure that organizes multiple Series together under a schema. Conceptually, it’s like a table in SQL or Excel.  
Rows represent records, and columns represent fields.

In [8]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 30, 35]
})
print(df)

shape: (3, 3)
┌─────┬─────────┬─────┐
│ id  ┆ name    ┆ age │
│ --- ┆ ---     ┆ --- │
│ i64 ┆ str     ┆ i64 │
╞═════╪═════════╪═════╡
│ 1   ┆ Alice   ┆ 25  │
│ 2   ┆ Bob     ┆ 30  │
│ 3   ┆ Charlie ┆ 35  │
└─────┴─────────┴─────┘


## Role of schemas and strict type consistency

One of Polars’ strengths is strict schema enforcement. Each column has a fixed data type, and Polars ensures all operations respect it. This prevents subtle bugs that arise in dynamically typed operations.

For instance, you cannot accidentally add a string column to an integer column without explicit conversion.

In [9]:
df = df.with_columns(
    (pl.col("age") + 5).alias("age_plus_5")
)
print(df)

shape: (3, 4)
┌─────┬─────────┬─────┬────────────┐
│ id  ┆ name    ┆ age ┆ age_plus_5 │
│ --- ┆ ---     ┆ --- ┆ ---        │
│ i64 ┆ str     ┆ i64 ┆ i64        │
╞═════╪═════════╪═════╪════════════╡
│ 1   ┆ Alice   ┆ 25  ┆ 30         │
│ 2   ┆ Bob     ┆ 30  ┆ 35         │
│ 3   ┆ Charlie ┆ 35  ┆ 40         │
└─────┴─────────┴─────┴────────────┘


Here, age is i64, and the result remains an integer. If you tried adding a string column, Polars would raise an error instead of silently failing.

## Eager vs. Lazy Execution

In [10]:
# Eager execution

df = pl.DataFrame({"x": [1, 2, 3]})
print(df.select(pl.col("x") * 2))  # eager: runs instantly

shape: (3, 1)
┌─────┐
│ x   │
│ --- │
│ i64 │
╞═════╡
│ 2   │
│ 4   │
│ 6   │
└─────┘


In [11]:
# Lazy execution

lazy_df = pl.DataFrame({"x": [1, 2, 3]}).lazy()
result = lazy_df.select(pl.col("x") * 2).collect()  # execute on collect()
print(result)

shape: (3, 1)
┌─────┐
│ x   │
│ --- │
│ i64 │
╞═════╡
│ 2   │
│ 4   │
│ 6   │
└─────┘
